# Wizard Model Fine-tuning with QLoRA
This notebook implements fine-tuning for Wizard models using 4-bit quantization and LoRA.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

In [ ]:
# Configuration
max_length = 128

# Model Configuration
model_type = "wizard7"  # Options: wizard7, wizard13, falcon, llama
dataset_type = "squad"  # Options: squad, reddit, reddit_negative

# LoRA Parameters
lora_alpha = 16
lora_dropout = 0.1
lora_r = 16
lora_bias = "all"

# Training Parameters
output_dir = "outputs_squad"
learning_rate = 0.00005
per_device_train_batch_size = 8
per_device_eval_batch_size = 8
gradient_accumulation_steps = 2
warmup_steps = 5
save_steps = 100
logging_steps = 25

In [ ]:
# Setup model and tokenizer
def get_model_path(model_type):
    if model_type == "wizard13":
        return "WizardLM/WizardLM-13B-V1.2"
    elif model_type == "wizard7":
        return "TheBloke/wizardLM-7B-HF"
    elif model_type == "falcon":
        return "tiiuae/falcon-7b"
    else:
        return "meta-llama/Llama-2-7b-hf"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

# Load model and tokenizer
model_path = get_model_path(model_type)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    device_map="auto",
    quantization_config=bnb_config,
    cache_dir="./models",
)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True,
    cache_dir="./models",
)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load and prepare dataset
if dataset_type == "squad":
    dataset = load_dataset("squad")
elif dataset_type == "reddit":
    dataset = load_dataset("gmongaras/reddit_political_2019_Feb", cache_dir="./datasets")
else:
    dataset = load_dataset("gmongaras/reddit_negative", cache_dir="./datasets")

# Prepare the dataset (example for squad)
def prepare_squad_dataset(examples):
    texts = [f"#### Human: {q}#### Assistant: {a}" 
             for q, a in zip(examples['question'], examples['answers']['text'])]
    
    encoded = tokenizer(texts, max_length=max_length-1, truncation=True, padding="max_length")
    encoded["labels"] = encoded["input_ids"].copy()
    
    return encoded

# Process dataset
processed_dataset = dataset.map(prepare_squad_dataset, batched=True)

# Split dataset
train_size = int(len(processed_dataset['train']) * 0.8)
train_dataset = processed_dataset['train'].select(range(train_size))
eval_dataset = processed_dataset['train'].select(range(train_size, len(processed_dataset['train'])))

In [ ]:
# Configure LoRA
lora_target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
]

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias=lora_bias,
    task_type="CAUSAL_LM",
    target_modules=lora_target_modules
)

model = get_peft_model(model, peft_config)

In [ ]:
# Configure training
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=learning_rate,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_steps=warmup_steps,
    save_steps=save_steps,
    logging_steps=logging_steps,
    evaluation_strategy="steps",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

# Start training
trainer.train()

In [ ]:
# Save the model
trainer.save_model("final_model")